In [2]:
import json
import os
import h5py
import numpy as np
import tensorflow as tf

2026-02-24 13:46:11.454531: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 13:46:11.626800: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 13:46:13.720329: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
import tensorflow as tf

path = "/scratch/mnhagen/datasets/incompressible_euler/cylinder_flow/train.tfrecord"

it = tf.compat.v1.io.tf_record_iterator(path)  # simple sequential reader
raw = next(it)
print("raw bytes:", len(raw))

ex = tf.train.Example()
ex.ParseFromString(raw)
print("num features:", len(ex.features.feature))
print("keys:", list(ex.features.feature.keys())[:50])  # first 50 keys

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`
raw bytes: 13572061
num features: 5
keys: ['velocity', 'mesh_pos', 'cells', 'pressure', 'node_type']


In [11]:
import os, json
import numpy as np
import h5py
import tensorflow as tf

DATASET_DIR = "/scratch/mnhagen/datasets/incompressible_euler/cylinder_flow"
OUT_DIR = "/scratch/mnhagen/datasets/incompressible_euler"
SPLIT = "valid"  # change to "valid" and "test" later

TFRECORD_PATH = os.path.join(DATASET_DIR, f"{SPLIT}.tfrecord")
META_PATH = os.path.join(DATASET_DIR, "meta.json")
OUT_H5 = os.path.join(OUT_DIR, f"{SPLIT}.h5")

with open(META_PATH, "r") as f:
    meta = json.load(f)

# All features are bytes blobs (len=1), as your inspection showed.
feature_description = {
    "velocity":  tf.io.FixedLenFeature([], tf.string),
    "mesh_pos":  tf.io.FixedLenFeature([], tf.string),
    "cells":     tf.io.FixedLenFeature([], tf.string),
    "pressure":  tf.io.FixedLenFeature([], tf.string),
    "node_type": tf.io.FixedLenFeature([], tf.string),
}

def parse_example(serialized):
    ex = tf.io.parse_single_example(serialized, feature_description)

    # Decode raw bytes into tensors
    mesh_pos  = tf.io.decode_raw(ex["mesh_pos"], tf.float32)   # (N*2,)
    velocity  = tf.io.decode_raw(ex["velocity"], tf.float32)   # (T*N*2,)
    pressure  = tf.io.decode_raw(ex["pressure"], tf.float32)   # (T*N,)
    cells     = tf.io.decode_raw(ex["cells"], tf.int32)        # (C*3,)
    node_type = tf.io.decode_raw(ex["node_type"], tf.int32)    # (N,)

    return {
        "mesh_pos": mesh_pos,
        "velocity": velocity,
        "pressure": pressure,
        "cells": cells,
        "node_type": node_type,
    }

ds = tf.data.TFRecordDataset([TFRECORD_PATH]).map(parse_example)

def infer_and_reshape(sample):
    pos_flat = sample["mesh_pos"]
    node_type = sample["node_type"]
    cells_flat = sample["cells"]
    vel_flat = sample["velocity"]
    pres_flat = sample["pressure"]

    N = int(node_type.shape[0])

    if int(pos_flat.shape[0]) != N * 2:
        raise ValueError(f"mesh_pos length {int(pos_flat.shape[0])} != N*2 ({N*2})")

    if int(cells_flat.shape[0]) % 3 != 0:
        raise ValueError(f"cells length {int(cells_flat.shape[0])} not divisible by 3")
    C = int(int(cells_flat.shape[0]) // 3)

    if int(vel_flat.shape[0]) % (N * 2) != 0:
        raise ValueError(f"velocity length {int(vel_flat.shape[0])} not divisible by N*2 ({N*2})")
    T = int(int(vel_flat.shape[0]) // (N * 2))

    if int(pres_flat.shape[0]) != T * N:
        raise ValueError(f"pressure length {int(pres_flat.shape[0])} != T*N ({T*N})")

    pos = tf.reshape(pos_flat, (N, 2))
    cells = tf.reshape(cells_flat, (C, 3))
    vel = tf.reshape(vel_flat, (T, N, 2))
    pres = tf.reshape(pres_flat, (T, N))

    return N, C, T, pos, cells, node_type, vel, pres

with h5py.File(OUT_H5, "w") as h5:
    # meta_json: your np.bytes_ choice is fine; this is simplest:
    h5.create_dataset("meta_json", data=json.dumps(meta).encode("utf-8"))

    for i, sample in enumerate(ds):
        N, C, T, pos, cells, node_type, vel, pres = infer_and_reshape(sample)

        # to numpy
        pos_np = pos.numpy().astype(np.float32)
        cells_np = cells.numpy().astype(np.int64)      # indices nice as int64 for torch
        node_type_np = node_type.numpy().astype(np.int32)
        vel_np = vel.numpy().astype(np.float32)
        pres_np = pres.numpy().astype(np.float32)

        g = h5.create_group(f"sample_{i:06d}")
        g.create_dataset("pos", data=pos_np, compression="gzip")
        g.create_dataset("cells", data=cells_np, compression="gzip")
        g.create_dataset("node_type", data=node_type_np, compression="gzip")
        g.create_dataset("vel", data=vel_np, compression="gzip")
        g.create_dataset("pressure", data=pres_np, compression="gzip")

        g.attrs["N"] = int(N)
        g.attrs["C"] = int(C)
        g.attrs["T"] = int(T)

        if i % 100 == 0:
            print(f"wrote {i} (N={int(N)}, C={int(C)}, T={int(T)})")

print("Done:", OUT_H5)

wrote 0 (N=1896, C=3558, T=600)
Done: /scratch/mnhagen/datasets/incompressible_euler/valid.h5
